# Week 03 — Анализ цен на дома (House Prices)

**Источник:** Kaggle, конкурс *House Prices — Advanced Regression Techniques*, автор/организатор: Kaggle. [Страница данных](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data). **Файл:** `train.csv`. **Дата скачивания:** впиши свою дату после скачивания. **Условия использования:** проверь правила конкурса Kaggle; исходный CSV в эту работу не включён.

**Одна строка** — один проданный дом в Ames, Iowa (США). Цены указаны в долларах США, площадь — в квадратных футах (sq ft).

### Словарь основных столбцов
| Столбец | Значение |
|---|---|
| Id | идентификатор записи |
| SalePrice | цена продажи, USD |
| GrLivArea | жилая площадь над землёй, sq ft |
| LotArea | площадь участка, sq ft |
| Neighborhood | район (категория) |
| OverallQual | оценка общего качества от 1 до 10 |
| GarageYrBlt | год постройки гаража; может отсутствовать |
| LotFrontage | длина границы участка вдоль улицы, футы; может отсутствовать |

**Вопросы исследования:** 1) Какова типичная цена дома? 2) В каких районах медианная цена выше (среди районов с достаточным числом домов)? 3) Как различаются цены домов с большой и небольшой жилой площадью?

**Кому полезно:** покупателям и агентам недвижимости для предварительного сравнения домов. Анализ описательный, не является точной оценкой стоимости конкретного дома.

## 1. Загрузка и знакомство


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

file_path = Path('/Users/sultankadyr/VScode/ml-ai-course/Week4/House_Prices_Week03_Project/train.csv')
if not file_path.exists():
    file_path = Path('train.csv')
if not file_path.exists():
    raise FileNotFoundError('Скачай train.csv с Kaggle и положи рядом с ноутбуком или в data/')
raw_df = pd.read_csv(file_path)
df = raw_df.copy()  # сохраняем оригинал без изменений
print('Размер:', df.shape)
print('Столбцы:', df.columns.tolist())
print('Типы:', df.dtypes.value_counts())
df.info()

Размер: (1460, 81)
Столбцы: ['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'Screen

In [3]:
display(df.head(1))
display(df.head(5))
display(df.head(10))
display(df.tail(5))
display(df.sample(5, random_state=42))

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
5,6,50,RL,85.0,14115,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,7,20,RL,75.0,10084,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,307000
7,8,60,RL,NaN,10382,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,Shed,350,11,2009,WD,Normal,200000
8,9,50,RM,51.0,6120,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2008,WD,Abnorml,129900
9,10,190,RL,50.0,7420,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,1,2008,WD,Normal,118000


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125
1459,1460,20,RL,75.0,9937,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2008,WD,Normal,147500


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
892,893,20,RL,70.0,8414,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2006,WD,Normal,154500
1105,1106,60,RL,98.0,12256,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,325000
413,414,30,RM,56.0,8960,Pave,Grvl,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2010,WD,Normal,115000
522,523,50,RM,50.0,5000,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,10,2006,WD,Normal,159000
1036,1037,20,RL,89.0,12898,Pave,NaN,IR1,HLS,AllPub,...,0,NaN,NaN,NaN,0,9,2009,WD,Normal,315500


**Что объяснить:** `head()` показывает начало файла, а `sample()` — случайные строки из всей таблицы. `random_state=42` помогает получать одну и ту же выборку при повторном запуске. Посмотри, какие поля числовые, какие текстовые. Если размер не `(1460, 81)`, проверь выбранный CSV.

## 2. Выбор строк и столбцов

In [5]:
one_series = df['SalePrice']
one_dataframe = df[['SalePrice']]
print(type(one_series), type(one_dataframe))
display(df.iloc[:10, :3])  # первые 10 строк, первые 3 столбца
display(df.iloc[::5, :3].head(10))  # каждая пятая строка

display(df.loc[df['GrLivArea'] > 2000, ['Id', 'Neighborhood', 'GrLivArea', 'SalePrice']].head())
display(df.dropna(subset=['SalePrice']).nlargest(10, 'SalePrice')[['Id', 'SalePrice', 'GrLivArea']])
display(df.sort_values(['Neighborhood', 'SalePrice'], ascending=[True, False])[['Neighborhood', 'SalePrice']].head(10))

<class 'pandas.core.series.Series'> <class 'pandas.core.frame.DataFrame'>


,Id,MSSubClass,MSZoning
0,1,60,RL
1,2,20,RL
2,3,60,RL
3,4,70,RL
4,5,60,RL
5,6,50,RL
6,7,20,RL
7,8,60,RL
8,9,50,RM
9,10,190,RL


,Id,MSSubClass,MSZoning
0,1,60,RL
5,6,50,RL
10,11,20,RL
15,16,45,RM
20,21,60,RL
25,26,20,RL
30,31,70,C (all)
35,36,60,RL
40,41,20,RL
45,46,120,RL


,Id,Neighborhood,GrLivArea,SalePrice
4,5,NoRidge,2198,250000
7,8,NWAmes,2090,200000
11,12,NridgHt,2324,345000
20,21,NridgHt,2376,325300
35,36,NridgHt,2452,309000


,Id,SalePrice,GrLivArea
691,692,755000,4316
1182,1183,745000,4476
1169,1170,625000,3627
898,899,611657,2364
803,804,582933,2822
1046,1047,556581,2868
440,441,555000,2402
769,770,538000,3279
178,179,501837,2234
798,799,485000,3140


,Neighborhood,SalePrice
639,Blmngtn,264561
1394,Blmngtn,246578
559,Blmngtn,234000
851,Blmngtn,215000
1019,Blmngtn,213490
597,Blmngtn,194201
229,Blmngtn,192500
385,Blmngtn,192000
1023,Blmngtn,191000
1004,Blmngtn,181000


**Пояснение:** `iloc` выбирает по номеру позиции, `loc` — по названиям и условиям. `::5` означает шаг 5. `head(10).dropna()` может дать меньше 10 строк, если среди первых десяти были пропуски; для топа лучше сначала убрать пропуски нужного поля, потом выбрать 10 строк.

## 3. Качество данных и пропуски

In [6]:
missing = pd.DataFrame({
    'Пропусков': df.isna().sum(),
    'Процент': (df.isna().mean() * 100).round(2)
}).sort_values('Пропусков', ascending=False)
display(missing)
print('Полных дублей:', df.duplicated().sum())
print('Повторов Id:', df['Id'].duplicated().sum())
print('Цена <= 0:', (df['SalePrice'] <= 0).sum())
print('Жилая площадь <= 0:', (df['GrLivArea'] <= 0).sum())
print('Участок <= 0:', (df['LotArea'] <= 0).sum())

,Пропусков,Процент
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
...,...,...
ExterQual,0,0.00
Exterior2nd,0,0.00
Exterior1st,0,0.00
RoofMatl,0,0.00


Полных дублей: 0
Повторов Id: 0
Цена <= 0: 0
Жилая площадь <= 0: 0
Участок <= 0: 0


**Важно:** пропуск не всегда ошибка. Например, пустое значение в `GarageYrBlt` может означать отсутствие гаража. Мы не заменяем все пропуски нулями. Для демонстрации очистки заполним `LotFrontage` медианой **внутри района**, а остальные пропуски сохраним, поскольку их смысл нужно изучать отдельно. Не удаляем повторяющиеся строки автоматически: одинаковые признаки могут описывать разные дома.

In [ ]:
before_rows = len(df)
missing_before = df['LotFrontage'].isna().sum()
# Медиана по району; если весь район пустой — общая медиана
by_neighborhood = df.groupby('Neighborhood')['LotFrontage'].transform('median')
df['LotFrontage'] = df['LotFrontage'].fillna(by_neighborhood)
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
print('Строк до:', before_rows, '| после:', len(df))
print('LotFrontage: пропусков до:', missing_before, '| после:', df['LotFrontage'].isna().sum())
print('Все остальные пропуски оставлены для осмысленной обработки при необходимости.')

## 4. Описательная статистика

In [ ]:
numeric = ['SalePrice', 'GrLivArea']
display(df[numeric].describe().round(2))
for column in numeric:
    print('\n', column)
    print('Известно:', df[column].count(), 'Среднее:', round(df[column].mean(), 2),
          'Медиана:', round(df[column].median(), 2),
          'Минимум:', df[column].min(), 'Максимум:', df[column].max(),
          'Ст. отклонение:', round(df[column].std(), 2))
    print('Квантили:', df[column].quantile([0.25, 0.50, 0.75, 0.90]).round(2).to_dict())
print('\nЧисло домов по районам:')
display(df['Neighborhood'].value_counts(dropna=False).head(10))
print('Проценты от всех строк:')
display((df['Neighborhood'].value_counts(normalize=True, dropna=False) * 100).round(2).head(10))

**Пояснение:** `mean` — среднее, `median` — середина, `std` — разброс. Квантиль 90% (`p90`) — уровень, ниже или на котором находятся около 90% наблюдений. В процентах знаменатель — число всех строк, включая записи с пустой категорией.

**После запуска допиши своими словами:** средняя цена = ___ USD, медиана = ___ USD; они отличаются потому, что ___. p90 цены = ___ USD; в расчёте участвовало ___ известных цен.

## 5. Фильтры и сравнение групп

In [ ]:
large_good = df[(df['GrLivArea'] > 2000) & (df['OverallQual'] >= 7)]
selected_areas = df[df['Neighborhood'].isin(['NAmes', 'CollgCr'])]
print('Большая площадь и высокое качество:', len(large_good))
print('Два выбранных района:', len(selected_areas))

threshold = df['GrLivArea'].median()
small = df[df['GrLivArea'] <= threshold]  # равные порогу идут сюда
large = df[df['GrLivArea'] > threshold]
comparison = pd.DataFrame({
    'Группа': ['Площадь <= медианы', 'Площадь > медианы'],
    'Строк': [len(small), len(large)],
    'Известных цен': [small['SalePrice'].count(), large['SalePrice'].count()],
    'Средняя цена': [small['SalePrice'].mean(), large['SalePrice'].mean()],
    'Медианная цена': [small['SalePrice'].median(), large['SalePrice'].median()]
})
display(comparison.round(2))
print('Порог площади:', threshold)
# Если площадь отсутствует, такая строка не попадёт ни в одну группу.

### Мини-эксперимент: меняем порог

In [ ]:
for threshold_test in [1500, 2000]:
    group = df[df['GrLivArea'] > threshold_test]
    print('Площадь >', threshold_test, '| домов:', len(group),
          '| медианная цена:', round(group['SalePrice'].median(), 2))

**Объясни:** при повышении порога меняется состав группы, количество домов и медианная цена. Это описание именно этой выборки, а не доказательство причинной связи.

## 6. Новый признак и групповой отчёт

In [ ]:
# Цена за квадратный фут жилой площади (USD/sq ft)
# Нельзя делить на ноль: заменяем нулевую площадь на NaN
safe_area = df['GrLivArea'].replace(0, np.nan)
df['PricePerSqFt'] = df['SalePrice'] / safe_area

group_report = df.groupby('Neighborhood', dropna=False).agg(
    houses=('Id', 'size'),
    known_prices=('SalePrice', 'count'),
    mean_price=('SalePrice', 'mean'),
    median_price=('SalePrice', 'median'),
    max_price=('SalePrice', 'max')
).reset_index().sort_values('median_price', ascending=False)
display(group_report.round(2))
print('Сумма домов по группам:', group_report['houses'].sum())
print('Число строк:', len(df))
assert group_report['houses'].sum() == len(df)
print('Медианная цена за кв. фут:', round(df['PricePerSqFt'].median(), 2))

`size` считает все строки группы, `count` — только непустые цены. Если числа совпадают, значит в цене нет пропусков. Районы с малым числом домов нельзя уверенно сравнивать с крупными.

## 7. Три графика

### График 1. Распределение цен
Берём непустые цены из `df`. Гистограмма подходит для числовых значений. Сравним 10 и 25 интервалов (`bins`).

In [ ]:
prices = df['SalePrice'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # figsize — размер рисунка
for ax, bins in zip(axes, [10, 25]):
    ax.hist(prices, bins=bins, color='steelblue')  # color — цвет столбцов
    ax.set_title(f'Цены домов: bins={bins}')
    ax.set_xlabel('Цена, USD')
    ax.set_ylabel('Количество домов')
    ax.grid(axis='y', alpha=0.2)  # лёгкая сетка для чтения
plt.tight_layout()  # чтобы подписи не налезали друг на друга
plt.show()

**После запуска:** сравни: при 25 интервалах столбики уже, видны более мелкие особенности; сами данные не изменились. Запиши, где находится основная масса цен.

### График 2. Количество домов по районам
Берём все строки `df`; показываем только 10 самых частых районов, остальные не показаны.

In [ ]:
top_areas = df['Neighborhood'].value_counts(dropna=False).head(10)
ax = top_areas.plot.bar(figsize=(10, 4), color='darkorange', rot=45)  # rot поворачивает подписи
ax.set_title('10 районов с наибольшим числом домов')
ax.set_xlabel('Район')
ax.set_ylabel('Количество домов')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()
print('Самый частый район:', top_areas.index[0], '| домов:', top_areas.iloc[0])

**После запуска:** назови самый частый район и количество домов. Поворот подписей помогает прочитать названия.

### График 3. Медианная цена по районам
Берём `group_report`; для честного сравнения показываем только районы с минимум 20 домами.

In [ ]:
enough_houses = group_report[group_report['houses'] >= 20].sort_values('median_price')
print('Размеры групп:')
display(enough_houses[['Neighborhood', 'houses', 'median_price']].round(2))
ax = enough_houses.plot.barh(x='Neighborhood', y='median_price', figsize=(9, 7),
                              color='seagreen', legend=False)
ax.set_title('Медианная цена по районам (от 20 домов)')
ax.set_xlabel('Медианная цена, USD')
ax.set_ylabel('Район')
ax.set_xlim(left=0)
plt.tight_layout()
Path('output').mkdir(exist_ok=True)
plt.savefig('output/chart.png', dpi=150, bbox_inches='tight')  # до show()
plt.show()
print('Сохранено: output/chart.png')

**После запуска:** назови район с самой высокой медианной ценой среди показанных. Укажи, что районы с менее чем 20 домами исключены из этого графика, но не из основной таблицы.

## 8. Сохраняем результаты и проверяем

In [ ]:
Path('output').mkdir(exist_ok=True)
group_report.to_csv('output/group_report.csv', index=False)
loaded_report = pd.read_csv('output/group_report.csv')
print('Строк в отчёте:', len(loaded_report))
print('Колонки:', loaded_report.columns.tolist())
print('Проверка числа домов:', loaded_report['houses'].sum(), '==', len(df))
assert loaded_report['houses'].sum() == len(df)
print('Версии: pandas', pd.__version__, '| numpy', np.__version__, '| matplotlib', plt.matplotlib.__version__)

## 9. Выводы: ответы на три вопроса

In [ ]:
print('1) Типичная цена (медиана):', round(df['SalePrice'].median(), 2), 'USD')
if len(enough_houses):
    most_expensive = enough_houses.sort_values('median_price', ascending=False).iloc[0]
    print('2) Наибольшая медиана среди районов с >=20 домами:', most_expensive['Neighborhood'],
          '|', round(most_expensive['median_price'], 2), 'USD')
print('3) Медианная цена домов с площадью <= медианы:', round(small['SalePrice'].median(), 2), 'USD')
print('   Медианная цена домов с площадью > медианы:', round(large['SalePrice'].median(), 2), 'USD')

**Ограничения:** 1) данные относятся к определённому городу и периоду — нельзя автоматически переносить выводы на весь рынок; 2) связь площади и цены не доказывает, что только площадь определяет стоимость. **Следующий вопрос:** как меняется цена при одинаковой площади, но разном качестве дома?

**Перед защитой:** запусти все ячейки, запиши конкретные числа из вывода выше и объясни, почему использовал медиану вместо одного только среднего.

## 10. Журнал работы с ИИ и защита
Ниже — **шаблон**, заполни колонку «Как я проверил» только после реального запуска. Не выдавай неподтверждённые проверки за сделанные.

| Мой вопрос | Что подсказал ИИ | Как я проверил | Что понял |
|---|---|---|---|
| Как работать с настоящими пропусками? | Проверить процент и заполнить `LotFrontage` медианой района | **Заполни после запуска:** сравнил пропуски до/после | Почему не все NaN заменяют нулём |
| Как сравнить две группы домов? | Разделить по медиане площади и посчитать цены | **Заполни после запуска:** поменял порог 1500 на 2000 | Почему меняется размер выборки |
| Как построить понятный график? | Построить гистограмму с `bins=10` и `bins=25` | **Заполни после запуска:** сравнил два графика | Что делает `bins` |

**Как я использовал ИИ:** попросил помочь выбрать более сложный датасет с реальными пропусками, объяснить код простыми словами и предложить проверки. **Моя задача:** запустить ноутбук, проверить результаты, поменять параметр и уметь объяснить каждую ячейку.

### План защиты (5–7 минут)
1. **1 минута:** источник, строки, колонки, три вопроса.
2. **1 минута:** покажи таблицу пропусков и объясни, почему заполнил `LotFrontage`.
3. **2 минуты:** `describe()`, медиана цены, сравнение площадей, отчёт по районам.
4. **1 минута:** покажи график и объясни оси.
5. **1 минута:** разберись с ячейкой `groupby`, расскажи про `size` и `count`.
6. **30 секунд:** что подсказал ИИ и что проверил сам.

**Практика перед защитой:** попробуй поменять `bins=25` на `bins=15`, порог `2000` на `1800` и `head(10)` на `head(7)`; объясни, что изменилось.

**Финальная проверка:** Restart Kernel → Run All. Все ячейки должны выполниться по порядку.